# 🏷️ HANDS-ON · CLASE 3: FINE-TUNING PARA CLASIFICAR · RESUELTO

Vas a **afinar** un modelo para que lea una reseña de **Chida Café ☕** y diga si es **POSITIVA** o **NEGATIVA** — y a **demostrar con números** que mejoró.

> 🅰️ **Antes** (sin afinar): adivina, ~50% de aciertos.   →   🅱️ **Después** (afinado): ~90%+ de aciertos.

### 🗺️ El plan
`Instalar → Cargar modelo → Dataset de reseñas → Medir ANTES → LoRA → Entrenar → Medir DESPUÉS + careo → Gráfica`

> ✅ **No necesita GPU ni API keys** — corre en segundos. (Si tienes GPU, mejor, pero no hace falta.)

Corre cada celda con **▶**. 🚀

## 🔧 Paso 1 · Instalar
Las librerías de Hugging Face para entrenar.

In [ ]:
%%capture
!pip install -q transformers peft datasets
!pip uninstall -y torchao   # Colab trae una versión vieja que choca con peft; no lo usamos, lo quitamos

## 📥 Paso 2 · Carga un modelo que sabe leer (DistilBERT)
Usamos **DistilBERT multilingüe** (modelo abierto, entiende español) con una "cabecita" para clasificar en 2 opciones.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
logging.set_verbosity_error()

ID2LABEL = {0: "NEGATIVO", 1: "POSITIVO"}
nombre = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(nombre)
modelo = AutoModelForSequenceClassification.from_pretrained(
    nombre, num_labels=2, id2label=ID2LABEL, label2id={"NEGATIVO":0, "POSITIVO":1})
print("Modelo cargado ✅ :", nombre)

## 🎨 Paso 3 · El dataset (reseñas etiquetadas)
Aquí está el 90% del trabajo: reseñas con su etiqueta (**1 = positiva, 0 = negativa**). Y unas de **prueba** aparte (que el modelo NO verá al entrenar).

In [ ]:
from datasets import Dataset

resenas = [
    ('El café de olla está delicioso, el mejor de la zona', 1),
    ('Súper buen ambiente y el personal muy amable', 1),
    ('El Frappe Chido es una maravilla, ya soy cliente fiel', 1),
    ('Rápido, limpio y con un café increíble', 1),
    ('Me encanta venir a trabajar aquí, el wifi vuela y el café buenísimo', 1),
    ('Excelente atención, se nota que les importa el cliente', 1),
    ('Los postres están de rechupete y a buen precio', 1),
    ('Ambiente acogedor y baristas que saben lo que hacen', 1),
    ('Volveré sin duda, todo estuvo perfecto', 1),
    ('El mejor lugar para desayunar, todo fresco y rico', 1),
    ('Pésimo servicio, esperé 40 minutos por un café frío', 0),
    ('Carísimo y el café sabe a agua sucia', 0),
    ('El personal fue grosero y el lugar estaba sucio', 0),
    ('Muy decepcionado, el pedido llegó incompleto', 0),
    ('Nunca más, el café estaba quemado y amargo', 0),
    ('Mala experiencia, el baño apestaba y las mesas pegajosas', 0),
    ('Tardísimo y encima se equivocaron de orden', 0),
    ('El frappe estaba aguado y sin sabor, no lo recomiendo', 0),
    ('Ruidoso, incómodo y con un café mediocre', 0),
    ('Cobran de más y la atención es horrible', 0),
]

# Reseñas de PRUEBA (held-out): NO están en el entrenamiento → miden si de verdad aprendió
TEST = [
    ('Todo delicioso, el mejor café que he probado en años', 1),
    ('Qué rico el pan y qué linda la atención', 1),
    ('Volvería mil veces, me trataron increíble', 1),
    ('Una porquería, el café estaba tibio y sin sabor', 0),
    ('Fatal, me cobraron mal y fueron muy groseros', 0),
    ('No regreso, sucio y carísimo para lo que ofrecen', 0),
]

# Preparar el dataset de entrenamiento
train_ds = Dataset.from_dict({"text":[t for t,l in resenas], "labels":[l for t,l in resenas]})
def tok(b): return tokenizer(b["text"], truncation=True, padding="max_length", max_length=64)
train_ds = train_ds.map(tok, batched=True)
print(len(resenas), "reseñas de entrenamiento y", len(TEST), "de prueba ✅")

## 🅰️ Paso 4 · ¿Qué tan bien clasifica ANTES?
Definimos cómo preguntarle y medimos su **% de aciertos** sin afinar. Como la cabecita está sin entrenar, va a **adivinar** (~50%).

In [ ]:
def predecir(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, max_length=64).to(modelo.device)
    modelo.eval()
    with torch.no_grad():
        logits = modelo(**inputs).logits
    return ID2LABEL[int(logits.argmax(dim=-1))]

def aciertos():
    return round(100 * sum(predecir(t)==ID2LABEL[l] for t,l in TEST) / len(TEST))

antes_pred = {t: predecir(t) for t,l in TEST}
antes_acc = aciertos()
print("Aciertos ANTES:", antes_acc, "%")

## 🧩 Paso 5 · Ponle LoRA
LoRA entrena unas "perillas nuevas" + la cabecita de clasificación, sin tocar el resto del modelo. Barato y rápido.

In [ ]:
from peft import LoraConfig, get_peft_model

config_lora = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16,
                         target_modules=["q_lin","v_lin"], lora_dropout=0.05,
                         modules_to_save=["pre_classifier","classifier"])  # ¡entrena también la cabecita!
modelo = get_peft_model(modelo, config_lora)
modelo.print_trainable_parameters()

## 🏋️ Paso 6 · Entrena
Mostramos las reseñas varias veces para que aprenda. Tarda **segundos** (aunque sea sin GPU).

In [ ]:
from transformers import TrainingArguments, Trainer

# Clasificar es barato: entrenamos varias vueltas (sin riesgo de que se rompa) para que la cabecita aprenda bien.
args = TrainingArguments(output_dir="/content/salida", num_train_epochs=40,
                         per_device_train_batch_size=4, learning_rate=5e-4,
                         logging_steps=10, report_to="none")
trainer = Trainer(model=modelo, args=args, train_dataset=train_ds)
trainer.train()
print("¡Entrenado! 🎉  (mira la pérdida bajar arriba)")

## 🅱️ Paso 7 · ¿Qué tan bien clasifica DESPUÉS? (el momento 🤯)
Las **mismas** reseñas de prueba, ahora con el modelo afinado. Míralas **lado a lado**.

In [ ]:
despues_pred = {t: predecir(t) for t,l in TEST}
despues_acc = aciertos()
print("Aciertos DESPUÉS:", despues_acc, "%\n")

print("="*70)
for t,l in TEST:
    real = ID2LABEL[l]
    a, d = antes_pred[t], despues_pred[t]
    print("📝", t)
    print("   🅰️ ANTES  :", a, "✅" if a==real else "❌")
    print("   🅱️ DESPUÉS:", d, "✅" if d==real else "❌", " (real:", real+")")
    print("-"*70)

## 📊 Paso 8 · La gráfica (aquí se paga todo)

In [ ]:
import matplotlib.pyplot as plt

b = plt.bar(["Antes","Después"], [antes_acc, despues_acc], color=["#9aa0a6","#2FA39A"])
plt.ylim(0,109); plt.ylabel("% de aciertos"); plt.title("Clasificación: Antes vs Después", weight="bold")
for i,v in enumerate([antes_acc, despues_acc]): plt.text(i, v+2, f"{v}%", ha="center", weight="bold")
plt.show()
print(f"\n🏁 De {antes_acc}% a {despues_acc}% de aciertos con solo afinar. ✅")

## 🎯 Reto final · Hazlo TUYO
1. Cambia las `resenas` por reseñas de **tu** negocio o tema (mantén el balance: mismas positivas que negativas).
2. Pon en `TEST` reseñas nuevas (que **no** estén en el entrenamiento).
3. Corre todo y mira tu **antes vs después**.

Acabas de **afinar y evaluar** un clasificador, de punta a punta. 🏷️🔥